In [1]:
import pandas as pd
pd.set_option ('display.max_columns', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/datos_limpios.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50116 entries, 0 to 50115
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       50116 non-null  int64  
 1   pais                     50116 non-null  str    
 2   edad                     49723 non-null  float64
 3   año_nacimiento           49723 non-null  float64
 4   sexo                     50116 non-null  str    
 5   años_educacion           49213 non-null  float64
 6   estado_marital           25806 non-null  str    
 7   decil_ingresos           39688 non-null  float64
 8   satisfaccion_vida        49752 non-null  float64
 9   felicidad                49897 non-null  float64
 10  reuniones_sociales       49981 non-null  float64
 11  deporte                  49333 non-null  float64
 12  horas_trabajo            42220 non-null  float64
 13  problemas_sueño          49915 non-null  float64
 14  interes_politica         50021 no

In [4]:
(df.isna().mean() * 100).round(2).sort_values(ascending=False)

estado_marital             48.51
decil_ingresos             20.81
horas_trabajo              15.76
confianza_politicos         1.90
años_educacion              1.80
importancia_tradiciones     1.72
deporte                     1.56
edad                        0.78
año_nacimiento              0.78
generacion                  0.78
satisfaccion_vida           0.73
felicidad                   0.44
nivel_entendimiento         0.41
problemas_sueño             0.40
sinceridad                  0.30
reuniones_sociales          0.27
interes_politica            0.19
uso_internet                0.10
sexo                        0.00
ID                          0.00
pais                        0.00
dtype: float64

Como podemos observar, tenemos casi un 50% de nulos en la columna de estado civil. Además, hay un porcentaje bastante alto el el decil de ingresos (20%) y horas de trabajo (15%). En cuanto al último, se debe a que hemos borrado anteriormente los datos que parecían poco lógicos, ya que la gente no puede trabajar 24 horas sin descanso. 
Por otra parte, los nulos que tenemos para la edad y el año de nacimiento se podrán ser eliminados, porque en este trabajo nos vamos a basarse en el estudio de generaciones y no nos van a aportar la información. 


In [5]:
df_nulos = df.copy()

In [6]:
df_nulos.sample()

,ID,pais,edad,año_nacimiento,sexo,años_educacion,estado_marital,decil_ingresos,satisfaccion_vida,felicidad,reuniones_sociales,deporte,horas_trabajo,problemas_sueño,interes_politica,confianza_politicos,importancia_tradiciones,uso_internet,sinceridad,nivel_entendimiento,generacion
39346,56404,PL,46.0,1977.0,mujer,14.0,casado legalmente,NaN,9.0,9.0,3.0,2.0,NaN,1.0,2.0,5.0,6.0,5.0,casi nada,casi siempre,Gen X


In [7]:
df_nulos = df_nulos.dropna(subset=["edad"])

Para estado marital vamos a crear una categoria nueva - "desconocido"

In [9]:
df_nulos["estado_marital"] = df_nulos["estado_marital"].fillna("desconocido")

Para las columnas numéricas, vamos a rellenar los nulos con la mediana según a que generación pertenecen.

In [11]:
numeric_cols = df_nulos.select_dtypes(include="number").columns

for col in numeric_cols:
    df_nulos[col] = df_nulos[col].fillna(
        df_nulos.groupby("generacion")[col].transform("median")
    )

Las dos últimas columnas categoricas, que tienen pocos nulos, los vamos a rellenar con la moda.

In [22]:
object_cols = df_nulos.select_dtypes(include="object").columns

for col in object_cols:
    df_nulos[col] = df_nulos[col].fillna(df_nulos[col].mode()[0])

/var/folders/4w/6nrn1lr91snggw8k807rmyz40000gn/T/ipykernel_20072/3960791499.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_nulos.select_dtypes(include="object").columns


In [24]:
df_nulos.to_csv("../data/datos_finales.csv", index=False)